In [ ]:
import mne
import os
import os.path as op
import numpy as np
from mne.time_frequency import tfr_morlet
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
task = 'MI'

sit_rest_lst = []
stand_rest_lst = []
sit_me_lst = []
stand_me_lst = []

for s_id in range(1, 24, 1):
    if s_id == 5: continue

    s_id_formatted = f'{s_id:02}'
    path_to_data = op.join('..', '..', 'nas-workspace', 'data', f'{task}_processed', f'S{s_id_formatted}.fif')

    raw = mne.read_epochs(path_to_data, preload=True)

    sit_mi = raw['mi_sit_std']
    stand_mi = raw['mi_std_sit']

    sit_rest = raw['mi_r_sit']
    stand_rest = raw['mi_r_std']

    sit_rest_lst.append(sit_rest[0 : 79])
    stand_rest_lst.append(stand_rest[0 : 79])
    sit_me_lst.append(sit_mi[0 : 36])
    stand_me_lst.append(stand_mi[0 : 36])

In [ ]:
sfreq = raw.info['sfreq']

frequencies = np.arange(1,40,1)

n_cycles = frequencies/2

sit_rest_all_power = []

stand_rest_all_power = []

for epoch in sit_rest_lst:
    power = tfr_morlet(epoch, freqs=frequencies, n_cycles = n_cycles, return_itc = False)

    power.apply_baseline(baseline=(-1, 0), mode='logratio')

    sit_rest_all_power.append(power.data)

sit_rest_grand_average_power = np.mean(np.array(sit_rest_all_power), axis = 0)

for epoch in stand_rest_lst:
    power = tfr_morlet(epoch, freqs=frequencies, n_cycles = n_cycles, return_itc = False)

    power.apply_baseline(baseline=(-1, 0), mode='logratio')

    stand_rest_all_power.append(power.data)

stand_rest_grand_average_power = np.mean(np.array(stand_rest_all_power), axis = 0)

# Transition Condition MI

In [ ]:
sfreq = raw.info['sfreq']

frequencies = np.arange(1,40,1)

n_cycles = frequencies/2

sit_me_all_power = []

stand_me_all_power = []

for epoch in sit_me_lst:
    power = tfr_morlet(epoch, freqs=frequencies, n_cycles = n_cycles, return_itc = False)

    power.apply_baseline(baseline=(-1, 0), mode='logratio')

    sit_me_all_power.append(power.data)

sit_me_grand_average_power = np.mean(np.array(sit_me_all_power), axis = 0)

for epoch in stand_me_lst:
    power = tfr_morlet(epoch, freqs=frequencies, n_cycles = n_cycles, return_itc = False)

    power.apply_baseline(baseline=(-1, 0), mode='logratio')

    stand_me_all_power.append(power.data)

stand_me_grand_average_power = np.mean(np.array(stand_me_all_power), axis = 0)

In [ ]:
task = 'mi'
imagery_task = 'stand_sit' #sit_stand or stand_sit
band = 'delta' #delta, theta, alpha, beta
# ================================================================================

band_dict = {'delta': [1,4], 'theta': [4,8], 'alpha': [8,13], 'beta': [13,30]}
sit_or_stand_dict = {'sit_stand': ['MI_SIT_STD', 'MI_R_SIT'], 'stand_sit': ['MI_STD_SIT', 'MI_R_STD']}
freq_band = band_dict[band]
freq_low = freq_band[0]
freq_high = freq_band[1]

sit_or_stand_label = sit_or_stand_dict[imagery_task]
upper_row_name = sit_or_stand_label[0]
lower_row_name = sit_or_stand_label[1]

time_ranges = [('Fixation', 'Fixation'), ('Visual Cue', 'Visual Cue'), (0,1), (1,2), (2,3), (3,4), (4,5)]
freq_idx = np.where((frequencies >= freq_low) & (frequencies <= freq_high))[0]

time_idx_min = 0  # Convert time to index. Starting time. Time idx 500 is second T=0. Time idx 0 is second T=-2
time_idx_max = 250  # Convert time to index. Ending time.

if band == 'beta':
    vmin = -0.10; vmax = 0.25
else:
    vmin = -0.05; vmax = 0.3

fig, axes = plt.subplots(2,7, figsize=(16,4))
axes = axes.ravel()

topomap_images = []

im = None

for (i, time_range, ax) in zip(range(0, len(axes.ravel())), time_ranges, axes[0:]):
    print(f'MI condition > time_idx_min : {time_idx_min} || time_idx_max : {time_idx_max}')
    
    if imagery_task == 'sit_stand':
        topo_data = sit_me_grand_average_power[:, freq_idx, time_idx_min:time_idx_max].mean(axis=2)
    elif imagery_task == 'stand_sit':
        topo_data = stand_me_grand_average_power[:, freq_idx, time_idx_min:time_idx_max].mean(axis=2)
        
    im, cn = mne.viz.plot_topomap(topo_data.mean(axis = 1), raw.info, cmap = 'Spectral_r', show = False, axes = ax, vlim=(vmin, vmax))
    topomap_images.append(im)

    time_idx_min = time_idx_min + 250
    time_idx_max = time_idx_max + 250
    
    if isinstance(time_range[0], int):
        axes[i].set_title(f'{time_range[0]}s to {time_range[1]}s')
    elif isinstance(time_range[0], str):
        axes[i].set_title(f'{time_range[0]}')

    axes[0].set_ylabel(upper_row_name, rotation=90, size='large')

time_idx_min = 0  # Convert time to index. Starting time. Time idx 500 is second T=0
time_idx_max = 250  # Convert time to index. Ending time.

for (i, time_range, ax) in zip(range(0, len(axes.ravel())), time_ranges, axes[7:]):

    print(f'Rest condition > time_idx_min : {time_idx_min} || time_idx_max : {time_idx_max}')

    
    if imagery_task == 'sit_stand':
        topo_data = sit_rest_grand_average_power[:, freq_idx, time_idx_min:time_idx_max].mean(axis=2)
    elif imagery_task == 'stand_sit':
        topo_data = stand_rest_grand_average_power[:, freq_idx, time_idx_min:time_idx_max].mean(axis=2)
        
    im, cn = mne.viz.plot_topomap(topo_data.mean(axis = 1), raw.info, cmap = 'Spectral_r', show = False, axes = ax, vlim=(vmin, vmax))
    topomap_images.append(im)

    time_idx_min = time_idx_min + 250
    time_idx_max = time_idx_max + 250

    axes[7].set_ylabel(lower_row_name, rotation=90, size='large')
    
plt.tight_layout()
fig.colorbar(im, ax = axes.ravel().tolist(), orientation='vertical', shrink =1).set_label('Power (dB)')
fig.patch.set_linewidth(1)
fig.patch.set_edgecolor('black')
plt.savefig(f'topomap_{task}_{imagery_task}_{band}_fixation_and_cue.pdf', dpi = 600, bbox_inches='tight')
plt.show()
